In [1]:
pip install optuna


   ---------------------------------------- 0.0/413.9 kB ? eta -:--:--
   -- ------------------------------------- 30.7/413.9 kB 1.3 MB/s eta 0:00:01
   -------- ------------------------------- 92.2/413.9 kB 1.3 MB/s eta 0:00:01
   ----------- -------------------------- 122.9/413.9 kB 901.1 kB/s eta 0:00:01
   ------------------ --------------------- 194.6/413.9 kB 1.1 MB/s eta 0:00:01
   ------------------------ --------------- 256.0/413.9 kB 1.2 MB/s eta 0:00:01
   ---------------------------------- ----- 358.4/413.9 kB 1.3 MB/s eta 0:00:01
   ---------------------------------------- 413.9/413.9 kB 1.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/247.4 kB ? eta -:--:--
   ---------------- ----------------------- 102.4/247.4 kB 3.0 MB/s eta 0:00:01
   ------------------------------- -------- 194.6/247.4 kB 2.4 MB/s eta 0:00:01
   ---------------------------------------- 247.4/247.4 kB 2.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import optuna
import numpy as np
import joblib

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score


In [3]:
RESULTS_DIR = r"C:\Users\PRAVEEN S\Desktop\BrainTumor-Cloud-AI\notebooks\results"

X_train = np.load(f"{RESULTS_DIR}/X_train_swinS.npy")
y_train = np.load(f"{RESULTS_DIR}/y_train_swinS.npy")

X_val = np.load(f"{RESULTS_DIR}/X_val_swinS.npy")
y_val = np.load(f"{RESULTS_DIR}/y_val_swinS.npy")

print("Train:", X_train.shape)
print("Val:", X_val.shape)


Train: (4440, 1408)
Val: (1111, 1408)


In [4]:
def objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 600),
        "max_depth": trial.suggest_int("max_depth", 4, 9),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08),
        "subsample": trial.suggest_float("subsample", 0.6, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 0.9),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 6),
        "gamma": trial.suggest_float("gamma", 0.0, 0.3),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 0.3),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 2.0),
        "eval_metric": "logloss",
        "random_state": 42,
        "n_jobs": -1
    }

    model = XGBClassifier(**params)

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    preds = model.predict(X_val)
    acc = accuracy_score(y_val, preds)

    return acc


In [5]:
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler())

study.optimize(objective, n_trials=30)

print("Best validation accuracy:", study.best_value)
print("Best parameters:")
study.best_params


[I 2026-01-30 21:38:05,802] A new study created in memory with name: no-name-d4af9ac4-b0da-4a4a-b6e2-4bce60829f1f
[I 2026-01-30 21:39:19,524] Trial 0 finished with value: 0.9090909090909091 and parameters: {'n_estimators': 219, 'max_depth': 7, 'learning_rate': 0.028648532843885653, 'subsample': 0.7356321597101949, 'colsample_bytree': 0.6433589386936482, 'min_child_weight': 1, 'gamma': 0.09301624022740572, 'reg_alpha': 0.130139086298706, 'reg_lambda': 1.9088781322305721}. Best is trial 0 with value: 0.9090909090909091.
[I 2026-01-30 21:40:50,353] Trial 1 finished with value: 0.9333933393339334 and parameters: {'n_estimators': 474, 'max_depth': 7, 'learning_rate': 0.04974219028418291, 'subsample': 0.7863377347197397, 'colsample_bytree': 0.8100391078095086, 'min_child_weight': 4, 'gamma': 0.2786216572963858, 'reg_alpha': 0.16725509106254008, 'reg_lambda': 1.3077032162848359}. Best is trial 1 with value: 0.9333933393339334.
[I 2026-01-30 21:41:55,446] Trial 2 finished with value: 0.9171917

Best validation accuracy: 0.9387938793879388
Best parameters:


{'n_estimators': 600,
 'max_depth': 5,
 'learning_rate': 0.043158364996775986,
 'subsample': 0.7127921112977201,
 'colsample_bytree': 0.6326837515364819,
 'min_child_weight': 2,
 'gamma': 0.14123083396419175,
 'reg_alpha': 0.12091950068202321,
 'reg_lambda': 0.5020608038332635}

In [6]:
best_params = study.best_params

best_model = XGBClassifier(
    **best_params,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

best_model.fit(X_train, y_train)

print("Optimized XGBoost trained ✅")


Optimized XGBoost trained ✅


In [7]:
X_test = np.load(f"{RESULTS_DIR}/X_test_swinS.npy")
y_test = np.load(f"{RESULTS_DIR}/y_test_swinS.npy")

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

from sklearn.metrics import classification_report, roc_auc_score

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("Test AUROC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))


Test Accuracy: 0.9502881844380403
Test AUROC: 0.9872701704087
              precision    recall  f1-score   support

           0       0.95      0.96      0.95       761
           1       0.95      0.94      0.94       627

    accuracy                           0.95      1388
   macro avg       0.95      0.95      0.95      1388
weighted avg       0.95      0.95      0.95      1388



In [8]:
MODEL_DIR = r"C:\Users\PRAVEEN S\Desktop\BrainTumor-Cloud-AI\models"

joblib.dump(best_model, f"{MODEL_DIR}/xgb_mobilevit_swinS_optuna.pkl")

print("Optimized model saved ✅")


Optimized model saved ✅
